# Image Classification Explainer Examples
This notebook demonstrates how to use the `ImageClassificationExplainer` from `xwhy` in four different scenarios:
1. Basic usage with a built-in model
2. Using a simple custom PyTorch model
3. Using a custom ResNet18 PyTorch model
4. Using a modern YOLO classification model (Ultralytics)


## Setup

Run the next cell first on Google Colab or Kaggle (internet access required). It installs XWhy from GitHub; you can skip it if XWhy is already installed locally.

In [ ]:
# Setup for Google Colab / Kaggle: install the latest XWhy from GitHub.
# Skip this cell if XWhy is already installed in your environment.
%pip install -q "xwhy @ git+https://github.com/Dependable-Intelligent-Systems-Lab/xwhy.git"

from importlib.metadata import version
print("xwhy", version("xwhy"))

# Fetch the sample images used below (if they are not already next to this notebook).
import glob, os, shutil, urllib.request

IMAGES_URL = "https://raw.githubusercontent.com/Dependable-Intelligent-Systems-Lab/xwhy/Examples/examples/Image_Classification_Examples"
for name in ["cat-and-dog.jpg", "cat-and-dog2.jpg"]:
    if os.path.exists(name):
        continue
    attached = glob.glob(f"/kaggle/input/**/{name}", recursive=True)  # Kaggle dataset, if attached
    if attached:
        shutil.copy(attached[0], name)
    else:
        urllib.request.urlretrieve(f"{IMAGES_URL}/{name}", name)
    print("Fetched", name)

## 1. Basic usage with a built-in model

In [ ]:
import xwhy
from xwhy import ImageClassificationExplainer

try:
    explainer = ImageClassificationExplainer(
        use_model_preprocess=False,
        use_embedding_model=True,
        segmentation_type="deeplabv3_resnet101",
        distance_type="wasserstein",
        classification_type="inception_v3",
    )
    
    result_basic = explainer.explain(instance="./cat-and-dog.jpg")
    print(result_basic.metrics)
    print("Explanation successful!")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    result_basic = None

In [ ]:
if result_basic:
    result_basic.plot()
    xwhy.plots.image_heatmap(result_basic)
    xwhy.plots.image(result_basic)
    xwhy.plots.plot_feature_bar_chart(result_basic)
    xwhy.plots.plot_feature_box_plot(result_basic)

In [ ]:
result = result_basic

In [ ]:
import xwhy

if result is not None:
    # 1. First plot: Positive only, hiding the rest of the image
    xwhy.plots.image_boundaries(
        result, 
        positive_only=True, 
        num_features=5, 
        hide_rest=True,
        title="Positive Regions (Hide Rest)"
    )

    # 2. Second plot: Positive only, showing the rest of the image
    xwhy.plots.image_boundaries(
        result, 
        positive_only=True, 
        num_features=5, 
        hide_rest=False,
        title="Positive Regions (Show Rest)"
    )

    # 3. Third plot: Top 10 features (positive & negative), showing the rest
    xwhy.plots.image_boundaries(
        result, 
        positive_only=False, 
        num_features=10, 
        hide_rest=False,
        title="Top 10 Regions (Positive & Negative)"
    )

    # 4.1. forth plot
    xwhy.plots.image_regions(result, positive_only=True, num_features=5, title="Positive Regions")
    xwhy.plots.image_regions(result, positive_only=False, num_features=10, title="Positive/Negative Regions")

    # 4.2. Fourth plot: Side-by-side subplots (Positive vs Positive/Negative)
    xwhy.plots.image_regions_side_by_side(result, num_features_pos=5, num_features_all=10)

In [ ]:
import matplotlib.pyplot as plt
from skimage.color import label2rgb
import xwhy
from xwhy import ImageClassificationExplainer

try:
    explainer = ImageClassificationExplainer(
        use_model_preprocess=False,
        use_embedding_model=True,
        segmentation_type="deeplabv3_resnet101",
        distance_type="wasserstein",
        classification_type="inception_v3",
    )
    
    result = explainer.explain(instance="./cat-and-dog2.jpg")
    print(result.metrics)
    print("Explanation successful!")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    result = None


if result is not None:
    # ========================================================
    # 1. YOUR XWHY PLOTS (With clean background fix)
    # ========================================================
    xwhy.plots.image_boundaries(
        result, positive_only=True, num_features=5, hide_rest=True, title="Positive Regions (Hide Rest)"
    )

    xwhy.plots.image_boundaries(
        result, positive_only=True, num_features=5, hide_rest=False, title="Positive Regions (Show Rest)"
    )

    xwhy.plots.image_boundaries(
        result, positive_only=False, num_features=10, hide_rest=False, title="Top 10 Regions (Positive & Negative)"
    )

    # Separated plot function calls
    xwhy.plots.image_regions(result, positive_only=True, num_features=5, title="Positive Regions")
    xwhy.plots.image_regions(result, positive_only=False, num_features=10, title="Positive/Negative Regions")

    # Side-by-side cleanly fixed xwhy plot
    xwhy.plots.image_regions_side_by_side(
        result, num_features_pos=5, num_features_all=10, 
        title1="xwhy: Positive", title2="xwhy: Pos/Neg (Clean BG)"
    )


    # ========================================================
    # 2. EXACT LIME TUTORIAL REPLICA (For comparison)
    # ========================================================
    print("Plotting original LIME behavior for comparison...")
    
    # Grab the mask/temp from our helper
    temp_pos, mask_pos = xwhy.plots.get_image_and_mask(result, positive_only=True, num_features=5, hide_rest=False)
    temp_all, mask_all = xwhy.plots.get_image_and_mask(result, positive_only=False, num_features=10, hide_rest=False)
    
    # To faithfully recreate the LIME bug, we switch our fixed '2' negative label back to '-1'
    lime_mask_all = mask_all.copy()
    lime_mask_all[lime_mask_all == 2] = -1

    # Recreate the LIME tutorial plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
    
    ax1.imshow(label2rgb(mask_pos, temp_pos, bg_label=0), interpolation='nearest')
    ax1.set_title('LIME Original: Positive Regions')
    
    # This is the exact code snippet from LIME utilizing the 3-mask trick
    ax2.imshow(label2rgb(3 - lime_mask_all, temp_all, bg_label=0), interpolation='nearest')
    ax2.set_title('LIME Original: Positive/Negative Regions')
    
    plt.show()

In [ ]:
import urllib.request
import xwhy
from xwhy import ImageClassificationExplainer

# 1. Download the exact sample image from the LIME tutorial repository
lime_image_url = "https://raw.githubusercontent.com/marcotcr/lime/master/doc/notebooks/data/cat_mouse.jpg"
image_path = "./cat_mouse.jpg"
print("Downloading LIME sample image...")
urllib.request.urlretrieve(lime_image_url, image_path)

# 2. Run the explainer on the LIME sample image
try:
    explainer = ImageClassificationExplainer(
        use_model_preprocess=False,
        use_embedding_model=True,
        segmentation_type="deeplabv3_resnet101",
        distance_type="wasserstein",
        classification_type="inception_v3",
    )
    
    print(f"Explaining {image_path}...")
    result = explainer.explain(instance=image_path)
    print("Explanation successful!")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    result = None

# 3. Plot the results
if result is not None:
    # See both positive and negative boundaries
    xwhy.plots.image_boundaries(
        result, 
        positive_only=False, 
        num_features=10, 
        hide_rest=False, 
        title="Top 10 Boundaries (Cat vs Mouse)"
    )

    # See the side-by-side colored regions
    xwhy.plots.image_regions_side_by_side(
        result, 
        num_features_pos=5, 
        num_features_all=10, 
        title1="Positive (Cat Parts)", 
        title2="Pos & Neg (Mouse parts in Red)"
    )

In [ ]:
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image
import xwhy
from xwhy import ImageClassificationExplainer

# 1. Fetch exactly the same dataset and do the exact same split as the LIME tutorial
print("Downloading/Loading Olivetti faces...")
faces = fetch_olivetti_faces()
X_train, X_test, y_train, y_test = train_test_split(faces.images, faces.target, test_size=0.2, random_state=0)

# 2. Save X_test[0] (which corresponds to y_test[0]) as a JPEG so your explainer can load it
img_array = (X_test[0] * 255).astype(np.uint8)
img = Image.fromarray(img_array).convert("RGB")
image_path = "./olivetti_test_0.jpg"
img.save(image_path)
print(f"Saved LIME tutorial exact test image to {image_path}")
print(f"Original ground truth label for this image: {y_test[0]}")

# 3. Run YOUR exact explainer on this exact image
try:
    explainer = ImageClassificationExplainer(
        use_model_preprocess=False,
        use_embedding_model=True,
        segmentation_type="deeplabv3_resnet101",
        distance_type="wasserstein",
        classification_type="inception_v3",
    )
    
    print("Explaining...")
    result = explainer.explain(instance=image_path)
    print(result.metrics)
    print("Explanation successful!")

except Exception as e:
    print(f"Error during pipeline execution: {e}")
    result = None

# 4. Plot using YOUR exact plot function
if result is not None:
    # Side-by-side cleanly fixed xwhy plot, just like the LIME notebook intended!
    xwhy.plots.image_regions_side_by_side(
        result, 
        num_features_pos=5, 
        num_features_all=10, 
        title1=f"Positive Regions for y_test[0]", 
        title2=f"Positive/Negative Regions for y_test[0]"
    )

In [ ]:
# 4. Plot using YOUR exact plot function
if result is not None:
    # Side-by-side cleanly fixed xwhy plot, just like the LIME notebook intended!
    xwhy.plots.image_regions_side_by_side(
        result, 
        num_features_pos=5, 
        num_features_all=50, 
        title1=f"Positive Regions for y_test[0]", 
        title2=f"Positive/Negative Regions for y_test[0]"
    )

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import transforms
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
import numpy as np
from PIL import Image

import xwhy
from xwhy.explainers.image import ImageClassificationExplainer

print("1. Fetching Olivetti dataset...")
faces = fetch_olivetti_faces()
X_train, X_test, y_train, y_test = train_test_split(faces.images, faces.target, test_size=0.2, random_state=0)

# 2. Define the Custom PyTorch Model for Faces (40 classes)
class FaceModel(nn.Module):
    def __init__(self, num_classes=40):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        # Input: 64x64 -> Pool1: 32x32 -> Pool2: 16x16
        self.fc1 = nn.Linear(32 * 16 * 16, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = FaceModel(num_classes=40)

# 3. Train the model quickly so it actually learns the conflicting facial features
print("2. Training the custom PyTorch model on Olivetti faces (this takes a few seconds)...")
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

# Convert X_train to PyTorch tensors (duplicate grayscale to 3 channels to match RGB JPG format)
X_train_t = torch.tensor(X_train).unsqueeze(1).repeat(1, 3, 1, 1).float()
y_train_t = torch.tensor(y_train).long()

model.train()
for epoch in range(20): # 20 epochs to guarantee it learns negative discriminative features
    optimizer.zero_grad()
    outputs = model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()
model.eval()

# 4. Save the EXACT LIME tutorial test face to JPG
print("3. Saving test image...")
img_array = (X_test[0] * 255).astype(np.uint8)
img = Image.fromarray(img_array).convert("RGB")
image_path = "./olivetti_test_0.jpg"
img.save(image_path)
print(f"   Saved {image_path}, True Label: {y_test[0]}")

# 5. Preprocessing for the Explainer (just resize to 64x64 and to tensor)
my_preprocess = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])

# 6. Run the xwhy Explainer!
print("4. Explaining the image with xwhy...")
try:
    explainer = ImageClassificationExplainer(
        custom_model=model,
        custom_preprocess=my_preprocess,
        use_embedding_model=False,
        use_segmentation_model=False,
        device="cpu" 
    )
    
    result = explainer.explain(instance=image_path)
    print("Explanation successful!")
    
    if result is not None:
        # Plot using our newly implemented side-by-side plot!
        xwhy.plots.image_regions_side_by_side(
            result, 
            num_features_pos=5, 
            num_features_all=10, 
            title1="Positive Regions", 
            title2="Positive & Negative Regions"
        )
except Exception as e:
    print(f"Error during pipeline execution: {e}")

In [ ]:
import torch
import torch.nn as nn
from torchvision import transforms
from sklearn.datasets import fetch_olivetti_faces
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import Normalizer
from sklearn.decomposition import PCA
import numpy as np
from PIL import Image
from skimage.color import rgb2gray, gray2rgb
import xwhy
from xwhy.explainers.image import ImageClassificationExplainer

print("1. Fetching Olivetti dataset exactly as LIME did...")
faces = fetch_olivetti_faces()

# Make each image color so the explainer works correctly (Marco's exact code)
X_vec = np.stack([gray2rgb(iimg) for iimg in faces.data.reshape((-1, 64, 64))], 0)
y_vec = faces.target.astype(np.uint8)

# Marco's exact split
X_train, X_test, y_train, y_test = train_test_split(X_vec, y_vec, train_size=0.70, random_state=0) 

print("2. Training Marco's EXACT LIME sklearn Pipeline...")
class PipeStep(object):
    def __init__(self, step_func):
        self._step_func=step_func
    def fit(self,*args):
        return self
    def transform(self,X):
        return self._step_func(X)

makegray_step = PipeStep(lambda img_list: [rgb2gray(img) for img in img_list])
flatten_step = PipeStep(lambda img_list: [img.ravel() for img in img_list])

# This is the exact model from the LIME notebook
simple_rf_pipeline = Pipeline([
    ('Make Gray', makegray_step),
    ('Flatten Image', flatten_step),
    ('Normalize', Normalizer()),
    ('PCA', PCA(25)),
    ('XGBoost', GradientBoostingClassifier())
])

simple_rf_pipeline.fit(X_train, y_train)

# ---------------------------------------------------------
# SECRET SAUCE: Wrap the Scikit-Learn pipeline in a PyTorch
# nn.Module so your `xwhy` ImageClassificationExplainer can run it!
# ---------------------------------------------------------
class SklearnLIMEWrapper(nn.Module):
    def __init__(self, pipeline):
        super().__init__()
        self.pipeline = pipeline

    def forward(self, x):
        # Convert PyTorch tensor back to Numpy array for sklearn
        x_np = x.permute(0, 2, 3, 1).cpu().numpy()
        
        # Get probabilities from Marco's model
        probas = self.pipeline.predict_proba(x_np)
        
        # xwhy expects logits, so we convert probabilities to log_probs
        log_probas = np.log(np.clip(probas, 1e-10, 1.0))
        return torch.tensor(log_probas, dtype=torch.float32, device=x.device)

wrapped_model = SklearnLIMEWrapper(simple_rf_pipeline)
# ---------------------------------------------------------

# Save the test image
img_array = (X_test[0] * 255).astype(np.uint8)
img = Image.fromarray(img_array)
image_path = "./olivetti_test_0.jpg"
img.save(image_path)
print(f"3. Saved LIME test image to {image_path}, True Label: {y_test[0]}")

# Preprocess for xwhy
my_preprocess = transforms.Compose([
    transforms.ToTensor()
])

print("4. Explaining the image with xwhy (using the EXACT LIME sklearn model)...")
explainer = ImageClassificationExplainer(
    custom_model=wrapped_model,
    custom_preprocess=my_preprocess,
    use_embedding_model=False,
    use_segmentation_model=False,
    device="cpu" 
)

result = explainer.explain(instance=image_path)
print("Explanation successful!")

# Plot using the exact Marco Ribeiro LIME math we restored in the previous step
xwhy.plots.image_regions_side_by_side(
    result, 
    num_features_pos=5, 
    num_features_all=10, 
    title1="Positive Regions", 
    title2="Positive & Negative Regions"
)

## 2. Custom Model Classification: Simple Model

In [ ]:
import torch
from torchvision import transforms
import torch.nn as nn
import torch.nn.functional as F

from xwhy.explainers.image import ImageClassificationExplainer

# 1. Define the custom model
class CustomVisionModel(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # Convolutional layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        
        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully connected layers
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x, 1) 
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits

# 2. Create an instance of the custom model
my_custom_model = CustomVisionModel(num_classes=10)

# 3. Define a custom preprocessing pipeline for this model
my_preprocess = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]) 
])

# 4. Initialize the explainer
try:
    explainer_simple = ImageClassificationExplainer(
        custom_model=my_custom_model,
        custom_preprocess=my_preprocess,
        use_embedding_model=False, 
        use_segmentation_model=False,
        device="cpu" 
    )

    image_path = "./cat-and-dog.jpg" 
    result_simple = explainer_simple.explain(instance=image_path)
    print(result_simple.metrics)

    print("Explanation Finished!")
    print("Top Feature Indices:", result_simple.top_features)
except Exception as e:
    print(f"Error: {e}")
    result_simple = None

## 3. Custom Model Classification: Resnet18

In [ ]:
import urllib.request
import torch
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from xwhy.explainers.image import ImageClassificationExplainer

# 1. Load pre-trained PyTorch model
weights = ResNet18_Weights.IMAGENET1K_V1
my_custom_model_resnet = resnet18(weights=weights)
my_custom_model_resnet.eval()

# 2. Define preprocessing pipeline
my_preprocess_resnet = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 3. Fetch ImageNet category names
url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
try:
    response = urllib.request.urlopen(url)
    imagenet_categories = [line.decode("utf-8").strip() for line in response.readlines()]
except Exception:
    imagenet_categories = None

# 4. Initialize Explainer
try:
    explainer_resnet = ImageClassificationExplainer(
        custom_model=my_custom_model_resnet,
        custom_preprocess=my_preprocess_resnet,
        categories=imagenet_categories,
        use_embedding_model=False,
        use_segmentation_model=False,
        device="cpu" 
    )

    # 5. Run the explanation
    image_path = "./cat-and-dog.jpg" 
    result_resnet = explainer_resnet.explain(
        instance=image_path,
        ground_truth_mask=None
    )

    print("\n--- Explanation Finished ---")
    print("Metrics:", result_resnet.metrics)
    print("Top Feature Indices:", result_resnet.top_features)
except Exception as e:
    print(f"Error: {e}")

## 4. YOLOv8/v11 Custom Model for `ImageClassificationExplainer`
This section demonstrates how to load a modern YOLO classification model (like YOLOv8-cls or YOLOv11-cls by Ultralytics) as a custom classification model and use it with the `ImageClassificationExplainer` in `xwhy`.


In [ ]:
!pip install -q xwhy ultralytics

In [ ]:
import requests
from PIL import Image

image_url = "https://ultralytics.com/images/zidane.jpg"
response = requests.get(image_url)
image_path_yolo = "sample_image.jpg"

with open(image_path_yolo, "wb") as f:
    f.write(response.content)

print(f"Sample image saved to {image_path_yolo}")
display(Image.open(image_path_yolo).resize((300, 300)))

In [ ]:
import torch
import torch.nn as nn
try:
    from ultralytics import YOLO
    yolo_available = True
except ImportError:
    yolo_available = False
    print("Ultralytics is not installed.")

if yolo_available:
    class YOLOClassificationWrapper(nn.Module):
        def __init__(self, model_name="yolo11n-cls.pt"):
            super().__init__()
            print(f"Loading {model_name}...")
            self.yolo = YOLO(model_name)
            self.model = self.yolo.model
            self.model.eval()
            
        def forward(self, x):
            out = self.model(x)
            if isinstance(out, (tuple, list)):
                return out[0]
            return out

    yolo_wrapper = YOLOClassificationWrapper("yolo11n-cls.pt")
    yolo_categories = list(yolo_wrapper.yolo.names.values())
    print(f"Loaded model with {len(yolo_categories)} classes.")

In [ ]:
from torchvision import transforms
my_preprocess_yolo = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

In [ ]:
if yolo_available:
    import xwhy
    from xwhy.explainers.image import ImageClassificationExplainer
    import warnings
    warnings.filterwarnings("ignore")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    yolo_wrapper.to(device)

    explainer_yolo = ImageClassificationExplainer(
        custom_model=yolo_wrapper,
        custom_preprocess=my_preprocess_yolo,
        categories=yolo_categories,
        use_embedding_model=False,
        use_segmentation_model=False,
        device=device 
    )
    
    print("Starting explanation... This might take a moment depending on your hardware.")
    try:
        result_yolo = explainer_yolo.explain(
            instance=image_path_yolo,
            ground_truth_mask=None
        )

        print("\n--- Explanation Finished ---")
        print("Metrics:", result_yolo.metrics)
        print("Top Feature Indices:", result_yolo.top_features)
        
        result_yolo.plot()
        xwhy.plots.image_heatmap(result_yolo)
        xwhy.plots.image(result_yolo)
    except Exception as e:
        print(f"Error: {e}")

# Visualization

In [ ]:
result = result_basic

In [ ]:
xwhy.plots.bar(result)

In [ ]:
xwhy.plots.image(result)

In [ ]:
xwhy.plots.image_heatmap(result)

In [ ]:
xwhy.plots.plot_feature_bar_chart(result)

In [ ]:
xwhy.plots.plot_feature_box_plot(result)